# ReferSplat: Referring Segmentation in 3D Gaussian Splatting
**ICML 2025 Oral**

이 노트북은 ReferSplat을 Google Colab 환경에서 실행하기 위한 전체 파이프라인입니다.

**파이프라인:**
1. 환경 설정 및 의존성 설치
2. 데이터셋 준비 (LERF-OVS)
3. 사전학습 3DGS 체크포인트 준비
4. ReferSplat 학습 (Stage 1)
5. 렌더링 (Inference)
6. 평가 (mIoU)

## 0. GPU 확인

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. 환경 설정

### 1-1. 경로 설정
**여기서 데이터 경로를 수정하세요.**

In [ ]:
import os

# ============================================================
# [수정 필요] 데이터 및 출력 경로 설정
# ============================================================
DATA_ROOT = "/content/data/lerf-ovs/Ref-lerf"  # LERF-OVS 데이터셋 루트 경로
OUTPUT_ROOT = "/content/output"                # 학습 결과 저장 경로 (Colab 로컬)
CHECKPOINT_ROOT = "/content/checkpoints"       # 사전학습 3DGS 체크포인트 경로
REPO_DIR = "/content/ReferSplat"               # 레포 클론 경로
GITHUB_REPO = "BAEJUNHAK/ReferSplat"          # GitHub 레포 (username/repo)

# Google Drive 저장 경로 (세션 끊겨도 유지)
DRIVE_SAVE_DIR = "/content/drive/MyDrive/ReferSplat_checkpoints"

# 사용할 장면 선택 (figurines, ramen, teatime, waldo_kitchen)
SCENE = "ramen"

# 사전학습 체크포인트 파일명 매핑 (HuggingFace 다운로드 구조에 맞춤)
CKPT_NAME_MAP = {
    "figurines": "figurineschkpnt30000.pth",
    "ramen": "ramenchkpnt30000.pth",
    "teatime": "teatimechkpnt30000.pth",
    "waldo_kitchen": "kitchenchkpnt30000.pth",
}

# 파생 경로 (자동 설정)
SCENE_DATA_PATH = os.path.join(DATA_ROOT, SCENE)
SCENE_OUTPUT_PATH = os.path.join(OUTPUT_ROOT, SCENE)
PRETRAINED_CKPT = os.path.join(CHECKPOINT_ROOT, CKPT_NAME_MAP[SCENE])

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)

print(f"Scene: {SCENE}")
print(f"Data path: {SCENE_DATA_PATH}")
print(f"Output path: {SCENE_OUTPUT_PATH}")
print(f"Pretrained checkpoint: {PRETRAINED_CKPT}")
print(f"Drive save dir: {DRIVE_SAVE_DIR}")
print(f"GitHub repo: {GITHUB_REPO}")

### 1-2. Google Drive 마운트 및 체크포인트 복원
이전 세션에서 학습한 체크포인트를 Drive에서 자동으로 불러옵니다.

In [ ]:
import glob, shutil
from google.colab import drive

# Google Drive 마운트
drive.mount('/content/drive')
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# Drive에 이전 세션의 체크포인트가 있으면 로컬로 복원
drive_scene_dir = os.path.join(DRIVE_SAVE_DIR, SCENE)
if os.path.exists(drive_scene_dir):
    drive_ckpts = sorted(glob.glob(os.path.join(drive_scene_dir, "chkpnt*.pth")))
    if drive_ckpts:
        os.makedirs(SCENE_OUTPUT_PATH, exist_ok=True)
        for ckpt in drive_ckpts:
            dst = os.path.join(SCENE_OUTPUT_PATH, os.path.basename(ckpt))
            if not os.path.exists(dst):
                print(f"Drive에서 복원: {os.path.basename(ckpt)}")
                shutil.copy2(ckpt, dst)
        print(f"복원 완료! 학습을 건너뛰고 바로 추론 가능합니다.")
    else:
        print("Drive에 저장된 체크포인트가 없습니다. 학습이 필요합니다.")
else:
    print("Drive에 저장된 체크포인트가 없습니다. 학습이 필요합니다.")

### 1-3. 레포지토리 클론 및 CUDA 서브모듈 빌드

In [ ]:
# 레포지토리 클론 (자신의 GitHub 레포에서)
%cd /content
if not os.path.exists(REPO_DIR):
    !git clone --recursive https://github.com/{GITHUB_REPO}.git
else:
    # 이미 클론된 경우 최신 변경사항 pull
    print("Repository already exists, pulling latest changes...")
    !cd {REPO_DIR} && git pull && git submodule update --init --recursive
%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
# 필수 패키지 설치
!pip install plyfile==0.8.1 jaxtyping open-clip-torch tensorboard -q

In [ ]:
# transformers 설치 (BERT 사용)
# 원본은 transformers==3.2.0이지만 Colab Python 3.10+와 호환 안됨
# BertTokenizer/BertModel API는 호환되므로 최신 버전 사용
!pip install transformers -q

In [ ]:
# CUDA 서브모듈 빌드: langsplat-rasterization
print("=" * 50)
print("Building langsplat-rasterization (CUDA)...")
print("=" * 50)
%cd {REPO_DIR}/submodules/langsplat-rasterization
!pip install --no-build-isolation .
%cd {REPO_DIR}

In [ ]:
# CUDA 서브모듈 빌드: simple-knn
print("=" * 50)
print("Building simple-knn (CUDA)...")
print("=" * 50)
%cd {REPO_DIR}/submodules/simple-knn
!pip install --no-build-isolation .
%cd {REPO_DIR}

In [ ]:
# segment-anything-langsplat 빌드
print("=" * 50)
print("Building segment-anything-langsplat...")
print("=" * 50)
%cd {REPO_DIR}/submodules/segment-anything-langsplat
!pip install --no-build-isolation .
%cd {REPO_DIR}

In [ ]:
# 빌드 검증
import sys
sys.path.insert(0, REPO_DIR)

try:
    import diff_gaussian_rasterization
    print("[OK] diff_gaussian_rasterization")
except ImportError as e:
    print(f"[FAIL] diff_gaussian_rasterization: {e}")

try:
    from simple_knn._C import distCUDA2
    print("[OK] simple_knn")
except ImportError as e:
    print(f"[FAIL] simple_knn: {e}")

try:
    from transformers import BertTokenizer, BertModel
    print("[OK] transformers (BERT)")
except ImportError as e:
    print(f"[FAIL] transformers: {e}")

## 2. 데이터셋 다운로드

LERF-OVS 데이터셋 (4개 장면: figurines, ramen, teatime, waldo_kitchen)

**다운로드 옵션:**
- HuggingFace: https://huggingface.co/datasets/FudanCVL/Ref-Lerf
- Baidu: https://pan.baidu.com/s/1D9yDNfUrK-d8eGO33Avkpg?pwd=ugs3

**데이터 구조 (각 장면별):**
```
<scene>/
├── images/              # RGB 이미지
├── sparse/0/            # COLMAP (cameras.bin, images.bin, points3D.bin)
├── json/
│   ├── train_json/      # frame_XXXXX.json (학습 어노테이션)
│   └── test_json/       # frame_XXXXX.json (테스트 어노테이션)
└── gt_mask/             # GT 세그멘테이션 마스크 (PNG)
```

In [ ]:
# ============================================================
# 데이터 다운로드 (HuggingFace) + 압축 해제 + 구조 정리
# ============================================================
import os

HF_DOWNLOAD_DIR = "/content/data/lerf-ovs"

# Step 1: HuggingFace에서 다운로드
if not os.path.exists(os.path.join(HF_DOWNLOAD_DIR, "Ref-lerf.zip")):
    !pip install huggingface_hub -q
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="FudanCVL/Ref-Lerf",
        repo_type="dataset",
        local_dir=HF_DOWNLOAD_DIR,
    )
    print("다운로드 완료!")
else:
    print("이미 다운로드됨. 스킵.")

# zip 파일 위치 확인 (HF 구조에 따라 다를 수 있음)
import glob
zip_base = HF_DOWNLOAD_DIR
ref_zip = glob.glob(os.path.join(HF_DOWNLOAD_DIR, "**/Ref-lerf.zip"), recursive=True)
if ref_zip:
    zip_base = os.path.dirname(ref_zip[0])
print(f"ZIP 파일 위치: {zip_base}")

# Step 2: Ref-lerf.zip 압축 해제
DATASET_DIR = os.path.join(zip_base, "Ref-lerf")
if not os.path.exists(os.path.join(DATASET_DIR, "ramen")):
    print("Ref-lerf.zip 압축 해제 중...")
    !cd {zip_base} && unzip -q -o Ref-lerf.zip
    print("완료!")
else:
    print("Ref-lerf.zip 이미 해제됨.")

# DATA_ROOT 업데이트
DATA_ROOT = DATASET_DIR
SCENE_DATA_PATH = os.path.join(DATA_ROOT, SCENE)
print(f"DATA_ROOT = {DATA_ROOT}")

# Step 3: 각 scene에 sparse/ 심볼릭 링크 생성 (distorted/sparse/ → sparse/)
# 코드가 <scene>/sparse/0/ 경로를 기대하기 때문
for scene_name in ["figurines", "ramen", "teatime", "waldo_kitchen"]:
    scene_dir = os.path.join(DATASET_DIR, scene_name)
    if not os.path.isdir(scene_dir):
        continue
    sparse_link = os.path.join(scene_dir, "sparse")
    distorted_sparse = os.path.join(scene_dir, "distorted", "sparse")
    if not os.path.exists(sparse_link) and os.path.exists(distorted_sparse):
        os.symlink(distorted_sparse, sparse_link)
        print(f"[{scene_name}] sparse/ -> distorted/sparse/ 링크 생성")
    elif os.path.exists(sparse_link):
        print(f"[{scene_name}] sparse/ 이미 존재")

# Step 4: ramen_masks.zip 해제 → ramen/gt_mask/
ramen_gt = os.path.join(DATASET_DIR, "ramen", "gt_mask")
ramen_zip = os.path.join(zip_base, "ramen_masks.zip")
if not os.path.exists(ramen_gt) and os.path.exists(ramen_zip):
    print("ramen_masks.zip 압축 해제 중...")
    !unzip -q -o {ramen_zip} -d /tmp/_tmp_ramen
    !mv /tmp/_tmp_ramen/gt_mask {ramen_gt}
    !rm -rf /tmp/_tmp_ramen
    print("ramen gt_mask 완료!")
elif os.path.exists(ramen_gt):
    print("ramen gt_mask 이미 존재.")

# Step 5: waldo_kitchen_mask.zip 해제 → waldo_kitchen/gt_mask/
waldo_gt = os.path.join(DATASET_DIR, "waldo_kitchen", "gt_mask")
waldo_zip = os.path.join(zip_base, "waldo_kitchen_mask.zip")
if not os.path.exists(waldo_gt) and os.path.exists(waldo_zip):
    print("waldo_kitchen_mask.zip 압축 해제 중...")
    !unzip -q -o {waldo_zip} -d /tmp/_tmp_waldo
    !mv /tmp/_tmp_waldo/waldo_kitchen/gt_mask {waldo_gt}
    !rm -rf /tmp/_tmp_waldo
    print("waldo_kitchen gt_mask 완료!")
elif os.path.exists(waldo_gt):
    print("waldo_kitchen gt_mask 이미 존재.")

# Step 6: 구조 확인
print("\n=== 데이터 구조 확인 ===")
for scene_name in ["figurines", "ramen", "teatime", "waldo_kitchen"]:
    scene_dir = os.path.join(DATASET_DIR, scene_name)
    has_images = os.path.isdir(os.path.join(scene_dir, "images"))
    has_sparse = os.path.exists(os.path.join(scene_dir, "sparse", "0", "cameras.bin"))
    has_json = os.path.isdir(os.path.join(scene_dir, "json"))
    has_gt = os.path.isdir(os.path.join(scene_dir, "gt_mask"))
    ok = lambda x: "OK" if x else "MISSING"
    print(f"  {scene_name}: images={ok(has_images)} sparse={ok(has_sparse)} json={ok(has_json)} gt_mask={ok(has_gt)}")

## 3. 체크포인트 다운로드

사전학습된 3DGS RGB 체크포인트가 필요합니다.

**다운로드:**
- Google Drive: https://drive.google.com/drive/folders/1z9O2FWwUlE29lSgLDj9Af7sv5ZQv6sc_
- HuggingFace: https://huggingface.co/FudanCVL/RefSplat

In [ ]:
# HuggingFace에서 체크포인트 다운로드
!pip install huggingface_hub -q
from huggingface_hub import snapshot_download

if not os.path.exists(os.path.join(CHECKPOINT_ROOT, "ramenchkpnt30000.pth")):
    snapshot_download(
        repo_id="FudanCVL/RefSplat",
        local_dir=CHECKPOINT_ROOT,
    )
    print("체크포인트 다운로드 완료!")
else:
    print("체크포인트 이미 존재. 스킵.")

# 확인
print("\n체크포인트 경로 확인:")
for scene_name, ckpt_name in CKPT_NAME_MAP.items():
    path = os.path.join(CHECKPOINT_ROOT, ckpt_name)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {scene_name}: {ckpt_name}")

## 4. 데이터 구조 검증

In [ ]:
import json
from pathlib import Path

def verify_scene(scene_path):
    """데이터셋 구조 검증"""
    checks = {
        "images/": os.path.isdir(os.path.join(scene_path, "images")),
        "sparse/0/": os.path.isdir(os.path.join(scene_path, "sparse", "0")),
        "cameras.bin": os.path.isfile(os.path.join(scene_path, "sparse", "0", "cameras.bin")),
        "images.bin": os.path.isfile(os.path.join(scene_path, "sparse", "0", "images.bin")),
        "points3D.bin": os.path.isfile(os.path.join(scene_path, "sparse", "0", "points3D.bin")),
        "gt_mask/": os.path.isdir(os.path.join(scene_path, "gt_mask")),
    }
    # json 폴더는 json/ 아래 또는 직접
    has_json = os.path.isdir(os.path.join(scene_path, "json")) or \
               os.path.isdir(os.path.join(scene_path, "train_json"))
    checks["json annotations"] = has_json

    print(f"\n--- Scene: {os.path.basename(scene_path)} ---")
    all_ok = True
    for name, ok in checks.items():
        status = "OK" if ok else "MISSING"
        print(f"  [{status}] {name}")
        if not ok:
            all_ok = False

    # 이미지 수 확인
    img_dir = os.path.join(scene_path, "images")
    if os.path.isdir(img_dir):
        n_images = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))])
        print(f"  Images: {n_images}")

    # GT mask 수 확인
    mask_dir = os.path.join(scene_path, "gt_mask")
    if os.path.isdir(mask_dir):
        n_masks = len([f for f in os.listdir(mask_dir) if f.endswith('.png')])
        print(f"  GT masks: {n_masks}")

    # 샘플 JSON 확인
    json_dir = os.path.join(scene_path, "json", "train_json")
    if not os.path.isdir(json_dir):
        json_dir = os.path.join(scene_path, "train_json")
    if os.path.isdir(json_dir):
        json_files = sorted([f for f in os.listdir(json_dir) if f.endswith('.json')])
        print(f"  Train JSONs: {len(json_files)}")
        if json_files:
            with open(os.path.join(json_dir, json_files[0])) as f:
                sample = json.load(f)
            print(f"  Sample JSON keys: {list(sample.keys())}")
            if 'object' in sample:
                obj = sample['object'][0]
                print(f"  Sample object keys: {list(obj.keys())}")
                if 'sentence' in obj:
                    print(f"  Sample sentences: {obj['sentence'][:2]}")

    return all_ok

# 선택한 장면 검증
if os.path.exists(SCENE_DATA_PATH):
    verify_scene(SCENE_DATA_PATH)
else:
    print(f"데이터 경로가 존재하지 않습니다: {SCENE_DATA_PATH}")

## 5. ReferSplat 학습 (Train)

**학습 흐름:**
1. 사전학습된 3DGS RGB 체크포인트 로드 (RGB 파라미터 freeze)
2. 각 Gaussian에 16차원 referring feature 추가
3. BERT로 텍스트 임베딩 → Position-aware Cross-Modal Interaction
4. BCE Loss + Contrastive Loss로 학습
5. 5 epoch, ~58분 (A6000 기준)

**주요 하이퍼파라미터:**
- referring feature dim: 16
- language_feature_lr: 0.0025
- mlp_lr / cross_attention_lr: 0.0001
- contrastive loss weight: 0.1
- top-τ schedule: 0.1 × 0.6^(iter/1000)

In [ ]:
%cd {REPO_DIR}
import glob

# 이미 학습된 체크포인트가 있으면 학습 건너뜀
existing_ckpts = sorted(glob.glob(os.path.join(SCENE_OUTPUT_PATH, "chkpnt*.pth")))
if existing_ckpts:
    print(f"학습된 체크포인트 발견! 학습을 건너뜁니다: {os.path.basename(existing_ckpts[-1])}")
    print("바로 섹션 6 (렌더링)으로 이동하세요.")
else:
    print(f"체크포인트 없음. 학습을 시작합니다.")
    !python train.py \
        -s {SCENE_DATA_PATH} \
        -m {SCENE_OUTPUT_PATH} \
        --start_checkpoint {PRETRAINED_CKPT}

In [ ]:
# 학습 완료 후 체크포인트를 Google Drive에 저장
import glob, shutil

ckpt_files = sorted(glob.glob(os.path.join(SCENE_OUTPUT_PATH, "chkpnt*.pth")))
if ckpt_files:
    drive_scene_dir = os.path.join(DRIVE_SAVE_DIR, SCENE)
    os.makedirs(drive_scene_dir, exist_ok=True)
    for ckpt in ckpt_files:
        dst = os.path.join(drive_scene_dir, os.path.basename(ckpt))
        if not os.path.exists(dst):
            print(f"Drive에 저장 중: {os.path.basename(ckpt)}")
            shutil.copy2(ckpt, dst)
    print(f"저장 완료! 경로: {drive_scene_dir}")
    print("다음 세션에서 자동으로 복원됩니다.")
else:
    print("저장할 체크포인트가 없습니다.")

### 5-1. (선택) 학습 모니터링

### 5-1. 학습 설정 확인
실제 학습에 사용된 하이퍼파라미터와 iteration 수를 확인합니다.

In [ ]:
import os, json, glob, torch

# --- 1. 코드 하이퍼파라미터 ---
print("=" * 60)
print("학습 하이퍼파라미터 (코드 기준)")
print("=" * 60)
print(f"  epoch_num:              5")
print(f"  language_feature_lr:    0.0025")
print(f"  mlp_lr:                 0.0001")
print(f"  cross_attention_lr:     0.0001")
print(f"  referring feature dim:  16")
print(f"  shared feature dim:     128")
print(f"  contrastive weight:     0.1")
print(f"  tau init:               0.1")
print(f"  tau decay:              ×0.6 every 2000 iter (min 0.005)")
print(f"  loss:                   BCE + 0.1 × contrastive")

# --- 2. 데이터셋 문장 수 (= iteration/epoch 계산) ---
print(f"\n{'=' * 60}")
print("데이터셋 문장 수 & Iteration 계산")
print("=" * 60)

for scene_name in ["ramen", "figurines", "teatime", "waldo_kitchen"]:
    scene_data = os.path.join(DATA_ROOT, scene_name)
    # train json
    json_dir = os.path.join(scene_data, "json", "train_json")
    if not os.path.isdir(json_dir):
        json_dir = os.path.join(scene_data, "train_json")
    if not os.path.isdir(json_dir):
        print(f"  {scene_name}: train_json 없음")
        continue
    total_sentences = 0
    n_frames = 0
    for jf in sorted(glob.glob(os.path.join(json_dir, "*.json"))):
        with open(jf) as f:
            data = json.load(f)
        for obj in data.get("object", []):
            total_sentences += len(obj.get("sentence", []))
        n_frames += 1
    iters_5 = total_sentences * 5
    print(f"  {scene_name:<20} train frames: {n_frames:<5} sentences/epoch: {total_sentences:<6} 5 epochs: {iters_5} iterations")

# --- 3. 체크포인트 실제 iteration 수 ---
print(f"\n{'=' * 60}")
print("체크포인트 저장된 iteration 수")
print("=" * 60)

for scene_name in ["ramen", "figurines", "teatime", "waldo_kitchen"]:
    scene_output = os.path.join(OUTPUT_ROOT, scene_name)
    ckpts = sorted(glob.glob(os.path.join(scene_output, "chkpnt*.pth")))
    if ckpts:
        for ckpt_path in ckpts:
            try:
                _, iteration = torch.load(ckpt_path, map_location="cpu", weights_only=False)
                print(f"  {scene_name:<20} {os.path.basename(ckpt_path):<35} iteration: {iteration}")
            except Exception as e:
                print(f"  {scene_name:<20} {os.path.basename(ckpt_path):<35} 로드 실패: {e}")
    else:
        print(f"  {scene_name:<20} 체크포인트 없음")

In [ ]:
# 저장된 체크포인트 확인
import glob

ckpt_files = sorted(glob.glob(os.path.join(SCENE_OUTPUT_PATH, "chkpnt*.pth")))
print(f"저장된 체크포인트 ({len(ckpt_files)}개):")
for f in ckpt_files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  {os.path.basename(f)} ({size_mb:.1f} MB)")

## 6. 렌더링 (Render)

학습된 모델로 테스트 뷰에서 세그멘테이션 마스크를 렌더링합니다.

**테스트 프레임 인덱스:**
- ramen: [6, 24, 60, 65, 81, 119, 128]
- figurines: [83, 97, 146, 179]
- teatime: [2, 25, 43, 107, 129, 140]
- waldo: [19, 35, 67, 105, 162]

In [ ]:
%%writefile {REPO_DIR}/render_colab.py
"""
Colab용 렌더링 스크립트 - 경로를 인자로 받음
"""
import re
import numpy as np
import torch
from scene import Scene
import os
from tqdm import tqdm
from os import makedirs
import torch.nn.functional as F
from gaussian_renderer import render
import torchvision
import random
from utils.general_utils import safe_state
from argparse import ArgumentParser
from arguments import ModelParams, PipelineParams, OptimizationParams
from gaussian_renderer import GaussianModel


def render_set(model_path, source_path, name, views, gaussians, pipeline, background, args):
    render_path = os.path.join(model_path, name, "renders")
    gts_path = os.path.join(model_path, name, "gt")
    render_npy_path = os.path.join(model_path, name, "renders_npy")
    gts_npy_path = os.path.join(model_path, name, "gt_npy")

    makedirs(render_npy_path, exist_ok=True)
    makedirs(gts_npy_path, exist_ok=True)
    makedirs(render_path, exist_ok=True)
    makedirs(gts_path, exist_ok=True)

    for idx, view in enumerate(tqdm(views, desc="Rendering progress")):
        for i in range(len(view.sentence)):
            sn = view.image_name
            number = re.findall(r'\d+', sn)
            number_int = int(number[0])
            output = render(view, gaussians, pipeline, background, args, sentence=view.sentence[i])
            rendering = output["language_feature_image"]
            rendering = torch.sigmoid(rendering)
            rendering = (rendering >= 0.5).float()
            gt = view.gt_mask[view.category[i]]
            fname = '{0:05d}'.format(number_int) + '{}'.format(view.category[i])
            np.save(os.path.join(render_npy_path, fname + ".npy"), rendering.permute(1, 2, 0).cpu().numpy())
            np.save(os.path.join(gts_npy_path, fname + ".npy"), gt.permute(1, 2, 0).cpu().numpy())
            torchvision.utils.save_image(rendering, os.path.join(render_path, fname + ".png"))
            torchvision.utils.save_image(gt, os.path.join(gts_path, fname + ".png"))


if __name__ == "__main__":
    random.seed(0)
    np.random.seed(0)
    torch.manual_seed(0)
    torch.cuda.manual_seed_all(0)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    parser = ArgumentParser()
    lp = ModelParams(parser)
    op = OptimizationParams(parser)
    pp = PipelineParams(parser)
    parser.add_argument("--checkpoint_file", type=str, required=True)
    parser.add_argument("--quiet", action="store_true")
    args = parser.parse_args()
    args.include_feature = True

    safe_state(args.quiet)

    with torch.no_grad():
        dataset = lp.extract(args)
        gaussians = GaussianModel(dataset.sh_degree)
        scene = Scene(dataset, gaussians, shuffle=False)

        (model_params, first_iter) = torch.load(args.checkpoint_file, map_location=f'cuda:{torch.cuda.current_device()}', weights_only=False)
        gaussians.restore(model_params, args, mode='test')

        bg_color = [1, 1, 1] if dataset.white_background else [0, 0, 0]
        background = torch.tensor(bg_color, dtype=torch.float32, device="cuda")

        render_set(dataset.model_path, dataset.source_path, "test_results",
                   scene.getTestCameras(), gaussians, pp.extract(args), background, args)

    print("Rendering complete!")

In [ ]:
import os, json, torch, numpy as np
from PIL import Image
from collections import defaultdict

def calculate_iou(pred_mask, gt_mask):
    intersection = torch.logical_and(pred_mask, gt_mask).sum().float()
    union = torch.logical_or(pred_mask, gt_mask).sum().float()
    if union == 0:
        return 0.0
    return (intersection / union).item()

def load_mask(file_path):
    mask = Image.open(file_path).convert("L")
    mask = np.array(mask)
    mask = torch.from_numpy(mask).float()
    return mask > 0

def load_sentences_from_json(data_path):
    """test_json에서 카테고리별 문장을 로드"""
    cat_to_sentences = defaultdict(list)
    json_dir = os.path.join(data_path, "json", "test_json")
    if not os.path.isdir(json_dir):
        json_dir = os.path.join(data_path, "test_json")
    if not os.path.isdir(json_dir):
        return cat_to_sentences
    for jf in sorted(os.listdir(json_dir)):
        if not jf.endswith(".json"):
            continue
        with open(os.path.join(json_dir, jf)) as f:
            data = json.load(f)
        for obj in data.get("object", []):
            cat = obj.get("category", "")
            for sent in obj.get("sentence", []):
                if sent not in cat_to_sentences[cat]:
                    cat_to_sentences[cat].append(sent)
    return cat_to_sentences

def evaluate_miou(render_dir, gt_dir, cat_sentences=None):
    """렌더링 결과와 GT 간 mIoU 계산 (문장 포함)"""
    iou_list = []
    details = []

    for filename in sorted(os.listdir(render_dir)):
        if not filename.endswith(".png"):
            continue
        render_path = os.path.join(render_dir, filename)
        gt_path = os.path.join(gt_dir, filename)
        if not os.path.exists(gt_path):
            continue

        pred_mask = load_mask(render_path)
        gt_mask = load_mask(gt_path)

        if gt_mask.sum() == 0:
            continue

        iou = calculate_iou(pred_mask, gt_mask)
        iou_list.append(iou)

        # 카테고리 추출 → 문장 매칭
        cat = filename.split(".png")[0]
        cat = ''.join([c for c in cat if not c.isdigit()]).strip()
        sentences = cat_sentences.get(cat, []) if cat_sentences else []

        details.append((filename, iou, cat, sentences))

    miou = np.mean(iou_list) if iou_list else 0.0
    return miou, details

# === 각 scene별 평가 실행 ===
ALL_SCENES = ["ramen", "waldo_kitchen"]
all_results = {}

for scene in ALL_SCENES:
    scene_data = os.path.join(DATA_ROOT, scene)
    scene_output = os.path.join(OUTPUT_ROOT, scene)
    render_dir = os.path.join(scene_output, "test_results", "renders")
    gt_dir = os.path.join(scene_output, "test_results", "gt")

    if not (os.path.exists(render_dir) and os.path.exists(gt_dir)):
        print(f"\n[SKIP] {scene}: 렌더링 결과 없음")
        continue

    cat_sentences = load_sentences_from_json(scene_data)
    miou, details = evaluate_miou(render_dir, gt_dir, cat_sentences)
    all_results[scene] = miou

    print(f"\n{'=' * 70}")
    print(f"Scene: {scene}  |  mIoU: {miou * 100:.2f}%")
    print(f"{'=' * 70}")
    print(f"{'File':<35} {'IoU':>6}  Referring Sentences")
    print("-" * 100)
    for fname, iou, cat, sentences in details:
        sent_str = sentences[0][:55] + "..." if sentences else "-"
        print(f"  {fname:<33} {iou*100:>5.1f}%  {sent_str}")
        for s in sentences[1:3]:
            print(f"  {'':33} {'':>6}  {s[:55]}...")

# === 전체 요약 ===
if all_results:
    avg = np.mean(list(all_results.values()))
    print(f"\n{'=' * 70}")
    print("Overall Summary:")
    for s, v in all_results.items():
        print(f"  {s}: {v*100:.2f}%")
    print(f"  Average: {avg*100:.2f}%")
    print(f"{'=' * 70}")

## 7. 평가 (mIoU)

In [ ]:
import os, json, torch, numpy as np
from PIL import Image
from collections import defaultdict

def calculate_iou(pred_mask, gt_mask):
    intersection = torch.logical_and(pred_mask, gt_mask).sum().float()
    union = torch.logical_or(pred_mask, gt_mask).sum().float()
    if union == 0:
        return 0.0
    return (intersection / union).item()

def load_mask(file_path):
    mask = Image.open(file_path).convert("L")
    mask = np.array(mask)
    mask = torch.from_numpy(mask).float()
    return mask > 0

def load_sentences_from_json(data_path):
    """test_json에서 카테고리별 문장을 로드"""
    cat_to_sentences = defaultdict(list)
    json_dir = os.path.join(data_path, "json", "test_json")
    if not os.path.isdir(json_dir):
        json_dir = os.path.join(data_path, "test_json")
    if not os.path.isdir(json_dir):
        return cat_to_sentences
    for jf in sorted(os.listdir(json_dir)):
        if not jf.endswith(".json"):
            continue
        with open(os.path.join(json_dir, jf)) as f:
            data = json.load(f)
        for obj in data.get("object", []):
            cat = obj.get("category", "")
            for sent in obj.get("sentence", []):
                if sent not in cat_to_sentences[cat]:
                    cat_to_sentences[cat].append(sent)
    return cat_to_sentences

def evaluate_miou(render_dir, gt_dir, cat_sentences=None):
    """렌더링 결과와 GT 간 mIoU 계산 (문장 포함)"""
    iou_list = []
    details = []

    for filename in sorted(os.listdir(render_dir)):
        if not filename.endswith(".png"):
            continue
        render_path = os.path.join(render_dir, filename)
        gt_path = os.path.join(gt_dir, filename)
        if not os.path.exists(gt_path):
            continue

        pred_mask = load_mask(render_path)
        gt_mask = load_mask(gt_path)

        if gt_mask.sum() == 0:
            continue

        iou = calculate_iou(pred_mask, gt_mask)
        iou_list.append(iou)

        # 카테고리 추출 → 문장 매칭
        cat = filename.split(".png")[0]
        cat = ''.join([c for c in cat if not c.isdigit()]).strip()
        sentences = cat_sentences.get(cat, []) if cat_sentences else []

        details.append((filename, iou, cat, sentences))

    miou = np.mean(iou_list) if iou_list else 0.0
    return miou, details

# 평가 실행
render_dir = os.path.join(SCENE_OUTPUT_PATH, "test_results", "renders")
gt_dir = os.path.join(SCENE_OUTPUT_PATH, "test_results", "gt")

if os.path.exists(render_dir) and os.path.exists(gt_dir):
    cat_sentences = load_sentences_from_json(SCENE_DATA_PATH)
    miou, details = evaluate_miou(render_dir, gt_dir, cat_sentences)

    print(f"\n{'=' * 70}")
    print(f"Scene: {SCENE}  |  mIoU: {miou * 100:.2f}%")
    print(f"{'=' * 70}")
    print(f"\nPer-query Results:")
    print(f"{'File':<35} {'IoU':>6}  Referring Sentences")
    print("-" * 100)
    for fname, iou, cat, sentences in details:
        sent_str = sentences[0][:55] + "..." if sentences else "-"
        print(f"  {fname:<33} {iou*100:>5.1f}%  {sent_str}")
        # 추가 문장이 있으면 표시
        for s in sentences[1:3]:
            print(f"  {'':33} {'':>6}  {s[:55]}...")
else:
    print(f"렌더링 결과가 없습니다. 먼저 렌더링을 실행하세요.")
    print(f"  Expected: {render_dir}")

## 8. 시각화

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

def visualize_results(render_dir, gt_dir, n_samples=4):
    """렌더링 결과 시각화 (Predicted vs GT)"""
    render_files = sorted([f for f in os.listdir(render_dir) if f.endswith('.png')])
    n_show = min(n_samples, len(render_files))

    if n_show == 0:
        print("시각화할 결과가 없습니다.")
        return

    fig, axes = plt.subplots(n_show, 2, figsize=(10, 4 * n_show))
    if n_show == 1:
        axes = axes[np.newaxis, :]

    for i, fname in enumerate(render_files[:n_show]):
        pred = np.array(Image.open(os.path.join(render_dir, fname)).convert("L"))
        gt_path = os.path.join(gt_dir, fname)
        gt = np.array(Image.open(gt_path).convert("L")) if os.path.exists(gt_path) else np.zeros_like(pred)

        axes[i, 0].imshow(pred, cmap='gray')
        axes[i, 0].set_title(f"Predicted: {fname}")
        axes[i, 0].axis('off')

        axes[i, 1].imshow(gt, cmap='gray')
        axes[i, 1].set_title(f"Ground Truth: {fname}")
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()

render_dir = os.path.join(SCENE_OUTPUT_PATH, "test_results", "renders")
gt_dir = os.path.join(SCENE_OUTPUT_PATH, "test_results", "gt")

if os.path.exists(render_dir):
    visualize_results(render_dir, gt_dir)
else:
    print("렌더링 결과가 없습니다.")

## 9. (선택) Two-Stage 최적화

Stage 1에서 렌더링된 마스크를 새로운 pseudo GT로 사용하여 재학습합니다.
논문 기준 ramen: 35.2 → 36.9 mIoU 향상.

In [ ]:
# 전체 장면 일괄 학습 + 렌더링 + 평가
# gt_mask가 제공된 장면만 실행 (figurines, teatime은 gt_mask 없음)

import glob, os, numpy as np

ALL_SCENES = ["ramen", "waldo_kitchen"]

CKPT_NAME_MAP = {
    "figurines": "figurineschkpnt30000.pth",
    "ramen": "ramenchkpnt30000.pth",
    "teatime": "teatimechkpnt30000.pth",
    "waldo_kitchen": "kitchenchkpnt30000.pth",
}

results = {}

for scene in ALL_SCENES:
    print(f"\n{'='*60}")
    print(f"Processing scene: {scene}")
    print(f"{'='*60}")

    scene_data = os.path.join(DATA_ROOT, scene)
    scene_output = os.path.join(OUTPUT_ROOT, scene)
    scene_ckpt = os.path.join(CHECKPOINT_ROOT, CKPT_NAME_MAP[scene])

    # 사전학습 체크포인트 확인
    if not os.path.exists(scene_ckpt):
        print(f"  [SKIP] 사전학습 체크포인트 없음: {scene_ckpt}")
        print(f"  섹션 3에서 체크포인트를 다시 다운로드하세요.")
        continue

    # 학습 (체크포인트 없으면)
    existing = sorted(glob.glob(os.path.join(scene_output, "chkpnt*.pth")))
    if existing:
        print(f"  체크포인트 발견, 학습 스킵: {os.path.basename(existing[-1])}")
    else:
        os.makedirs(scene_output, exist_ok=True)
        !cd {REPO_DIR} && python train.py -s {scene_data} -m {scene_output} --start_checkpoint {scene_ckpt}

    # 렌더링
    ckpt_files = sorted(glob.glob(os.path.join(scene_output, "chkpnt*.pth")))
    if ckpt_files:
        latest = ckpt_files[-1]
        !cd {REPO_DIR} && python render_colab.py -s {scene_data} -m {scene_output} --checkpoint_file {latest}

    # 평가
    r_dir = os.path.join(scene_output, "test_results", "renders")
    g_dir = os.path.join(scene_output, "test_results", "gt")
    if os.path.exists(r_dir) and os.path.exists(g_dir):
        miou, _ = evaluate_miou(r_dir, g_dir)
        results[scene] = miou
        print(f"  {scene} mIoU: {miou*100:.2f}%")
    else:
        print(f"  {scene} 렌더링 결과 없음")

# 최종 결과
if results:
    avg = np.mean(list(results.values()))
    print(f"\n{'='*60}")
    print("Final Results:")
    for s, v in results.items():
        print(f"  {s}: {v*100:.2f}%")
    print(f"  Average: {avg*100:.2f}%")
    print(f"{'='*60}")

## 10. 전체 장면 일괄 실행

4개 장면 모두 학습 + 평가하려면 아래를 실행하세요.

In [ ]:
# 전체 장면 일괄 학습 + 렌더링 + 평가
# 주의: 4개 장면 × ~58분 = 약 4시간 소요

import glob, os, numpy as np

ALL_SCENES = ["ramen", "figurines", "teatime", "waldo_kitchen"]

CKPT_NAME_MAP = {
    "figurines": "figurineschkpnt30000.pth",
    "ramen": "ramenchkpnt30000.pth",
    "teatime": "teatimechkpnt30000.pth",
    "waldo_kitchen": "kitchenchkpnt30000.pth",
}

results = {}

for scene in ALL_SCENES:
    print(f"\n{'='*60}")
    print(f"Processing scene: {scene}")
    print(f"{'='*60}")

    scene_data = os.path.join(DATA_ROOT, scene)
    scene_output = os.path.join(OUTPUT_ROOT, scene)
    scene_ckpt = os.path.join(CHECKPOINT_ROOT, CKPT_NAME_MAP[scene])

    # 학습 (체크포인트 없으면)
    existing = sorted(glob.glob(os.path.join(scene_output, "chkpnt*.pth")))
    if existing:
        print(f"  체크포인트 발견, 학습 스킵: {os.path.basename(existing[-1])}")
    else:
        os.makedirs(scene_output, exist_ok=True)
        !cd {REPO_DIR} && python train.py -s {scene_data} -m {scene_output} --start_checkpoint {scene_ckpt}

    # 렌더링
    ckpt_files = sorted(glob.glob(os.path.join(scene_output, "chkpnt*.pth")))
    if ckpt_files:
        latest = ckpt_files[-1]
        !cd {REPO_DIR} && python render_colab.py -s {scene_data} -m {scene_output} --checkpoint_file {latest}

    # 평가
    r_dir = os.path.join(scene_output, "test_results", "renders")
    g_dir = os.path.join(scene_output, "test_results", "gt")
    if os.path.exists(r_dir) and os.path.exists(g_dir):
        miou, _ = evaluate_miou(r_dir, g_dir)
        results[scene] = miou
        print(f"  {scene} mIoU: {miou*100:.2f}%")
    else:
        print(f"  {scene} 렌더링 결과 없음")

# 최종 결과
if results:
    avg = np.mean(list(results.values()))
    print(f"\n{'='*60}")
    print("Final Results:")
    for s, v in results.items():
        print(f"  {s}: {v*100:.2f}%")
    print(f"  Average: {avg*100:.2f}%")
    print(f"{'='*60}")

import os, glob, torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image

ALL_SCENES = ["ramen", "waldo_kitchen"]  # gt_mask가 있는 장면만

def analyze_size_vs_iou(render_dir, gt_dir):
    """객체 GT 마스크 면적 vs IoU 분석"""
    sizes = []
    ious = []
    labels = []

    for fname in sorted(os.listdir(render_dir)):
        if not fname.endswith(".png"):
            continue
        gt_path = os.path.join(gt_dir, fname)
        if not os.path.exists(gt_path):
            continue

        pred = torch.from_numpy(np.array(Image.open(os.path.join(render_dir, fname)).convert("L"))).float() > 0
        gt = torch.from_numpy(np.array(Image.open(gt_path).convert("L"))).float() > 0

        gt_area = gt.sum().item()
        if gt_area == 0:
            continue

        intersection = torch.logical_and(pred, gt).sum().float()
        union = torch.logical_or(pred, gt).sum().float()
        iou = (intersection / union).item() if union > 0 else 0.0

        total_pixels = gt.numel()
        area_ratio = gt_area / total_pixels * 100  # 전체 이미지 대비 %

        sizes.append(area_ratio)
        ious.append(iou * 100)
        # 객체 이름 추출 (00006bowl.png -> bowl)
        obj_name = fname.split(".png")[0]
        obj_name = ''.join([c for c in obj_name if not c.isdigit()])
        labels.append(obj_name)

    # Scatter plot
    fig, ax = plt.subplots(1, 1, figsize=(12, 7))
    scatter = ax.scatter(sizes, ious, c=ious, cmap='RdYlGn', s=60, alpha=0.7, edgecolors='black', linewidth=0.5)
    plt.colorbar(scatter, label='IoU (%)')

    # 객체별 라벨 (겹치지 않게 일부만)
    unique_labels = list(set(labels))
    for label in unique_labels:
        idxs = [i for i, l in enumerate(labels) if l == label]
        avg_size = np.mean([sizes[i] for i in idxs])
        avg_iou = np.mean([ious[i] for i in idxs])
        ax.annotate(label, (avg_size, avg_iou), fontsize=8, ha='center',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

    ax.set_xlabel('GT Mask Area (% of image)', fontsize=12)
    ax.set_ylabel('IoU (%)', fontsize=12)
    ax.set_title('Object Size vs Segmentation IoU', fontsize=14)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 크기 구간별 통계
    sizes_arr = np.array(sizes)
    ious_arr = np.array(ious)
    bins = [(0, 1, "Tiny (<1%)"), (1, 5, "Small (1-5%)"), (5, 15, "Medium (5-15%)"), (15, 100, "Large (>15%)")]
    print(f"\n{'구간':<20} {'개수':<8} {'평균 IoU':<12} {'표준편차':<12}")
    print("-" * 52)
    for lo, hi, name in bins:
        mask = (sizes_arr >= lo) & (sizes_arr < hi)
        if mask.sum() > 0:
            print(f"{name:<20} {mask.sum():<8} {ious_arr[mask].mean():<12.1f} {ious_arr[mask].std():<12.1f}")

    # 상관계수
    corr = np.corrcoef(sizes_arr, ious_arr)[0, 1]
    print(f"\n면적-IoU 상관계수 (Pearson): {corr:.3f}")

# 실행
for scene in ALL_SCENES:
    scene_output = os.path.join(OUTPUT_ROOT, scene)
    r_dir = os.path.join(scene_output, "test_results", "renders")
    g_dir = os.path.join(scene_output, "test_results", "gt")
    if os.path.exists(r_dir) and os.path.exists(g_dir):
        print(f"\n{'='*60}")
        print(f"Scene: {scene}")
        print(f"{'='*60}")
        analyze_size_vs_iou(r_dir, g_dir)

---
## 11. 문제점 분석 실험

### 11-1. 객체 크기별 성능 분석
GT 마스크의 면적(픽셀 수) 대비 IoU를 분석하여, 작은 객체일수록 성능이 떨어지는지 정량적으로 확인합니다.

In [ ]:
import os, glob, torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image

def analyze_size_vs_iou(render_dir, gt_dir):
    """객체 GT 마스크 면적 vs IoU 분석"""
    sizes = []
    ious = []
    labels = []

    for fname in sorted(os.listdir(render_dir)):
        if not fname.endswith(".png"):
            continue
        gt_path = os.path.join(gt_dir, fname)
        if not os.path.exists(gt_path):
            continue

        pred = torch.from_numpy(np.array(Image.open(os.path.join(render_dir, fname)).convert("L"))).float() > 0
        gt = torch.from_numpy(np.array(Image.open(gt_path).convert("L"))).float() > 0

        gt_area = gt.sum().item()
        if gt_area == 0:
            continue

        intersection = torch.logical_and(pred, gt).sum().float()
        union = torch.logical_or(pred, gt).sum().float()
        iou = (intersection / union).item() if union > 0 else 0.0

        total_pixels = gt.numel()
        area_ratio = gt_area / total_pixels * 100  # 전체 이미지 대비 %

        sizes.append(area_ratio)
        ious.append(iou * 100)
        # 객체 이름 추출 (00006bowl.png -> bowl)
        obj_name = fname.split(".png")[0]
        obj_name = ''.join([c for c in obj_name if not c.isdigit()])
        labels.append(obj_name)

    # Scatter plot
    fig, ax = plt.subplots(1, 1, figsize=(12, 7))
    scatter = ax.scatter(sizes, ious, c=ious, cmap='RdYlGn', s=60, alpha=0.7, edgecolors='black', linewidth=0.5)
    plt.colorbar(scatter, label='IoU (%)')

    # 객체별 라벨 (겹치지 않게 일부만)
    unique_labels = list(set(labels))
    for label in unique_labels:
        idxs = [i for i, l in enumerate(labels) if l == label]
        avg_size = np.mean([sizes[i] for i in idxs])
        avg_iou = np.mean([ious[i] for i in idxs])
        ax.annotate(label, (avg_size, avg_iou), fontsize=8, ha='center',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

    ax.set_xlabel('GT Mask Area (% of image)', fontsize=12)
    ax.set_ylabel('IoU (%)', fontsize=12)
    ax.set_title('Object Size vs Segmentation IoU', fontsize=14)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 크기 구간별 통계
    sizes_arr = np.array(sizes)
    ious_arr = np.array(ious)
    bins = [(0, 1, "Tiny (<1%)"), (1, 5, "Small (1-5%)"), (5, 15, "Medium (5-15%)"), (15, 100, "Large (>15%)")]
    print(f"\n{'구간':<20} {'개수':<8} {'평균 IoU':<12} {'표준편차':<12}")
    print("-" * 52)
    for lo, hi, name in bins:
        mask = (sizes_arr >= lo) & (sizes_arr < hi)
        if mask.sum() > 0:
            print(f"{name:<20} {mask.sum():<8} {ious_arr[mask].mean():<12.1f} {ious_arr[mask].std():<12.1f}")

    # 상관계수
    corr = np.corrcoef(sizes_arr, ious_arr)[0, 1]
    print(f"\n면적-IoU 상관계수 (Pearson): {corr:.3f}")

# 실행
for scene in ALL_SCENES:
    scene_output = os.path.join(OUTPUT_ROOT, scene)
    r_dir = os.path.join(scene_output, "test_results", "renders")
    g_dir = os.path.join(scene_output, "test_results", "gt")
    if os.path.exists(r_dir) and os.path.exists(g_dir):
        print(f"\n{'='*60}")
        print(f"Scene: {scene}")
        print(f"{'='*60}")
        analyze_size_vs_iou(r_dir, g_dir)

### 11-2. 객체 유형별 성능 분석 (투명/반사/일반)
투명 객체, 반사 객체, 일반 객체로 분류하여 3DGS 기반 방법의 근본적 한계를 분석합니다.

In [ ]:
import os, torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image

# 객체 속성 분류 (수동 태깅)
OBJECT_TYPE = {
    # 투명/반사 객체
    "glass of water": "transparent", "sake cup": "transparent",
    "dark cup": "transparent", "red cup": "transparent", "frog cup": "transparent",
    # 가늘고 작은 객체
    "chopsticks": "thin", "spoon": "thin", "knife": "thin",
    "black spoon": "thin", "plastic ladle": "thin", "spatula": "thin",
    # 텍스트/복잡한 표면
    "ottolenghi": "textured", "refrigerator": "textured",
    # 일반 불투명 객체
    "bowl": "opaque", "plate": "opaque", "nori": "opaque",
    "egg": "opaque", "eggs": "opaque", "corn": "opaque",
    "kamaboko": "opaque", "wavy noodles": "opaque", "hand": "opaque",
    "pot": "opaque", "ketchup": "opaque", "cabinet": "opaque",
    "pour-over vessel": "opaque", "stainless steel pot": "opaque",
}

def analyze_object_type(render_dir, gt_dir):
    """객체 유형별 성능 분석"""
    type_ious = {"transparent": [], "thin": [], "textured": [], "opaque": [], "unknown": []}

    for fname in sorted(os.listdir(render_dir)):
        if not fname.endswith(".png"):
            continue
        gt_path = os.path.join(gt_dir, fname)
        if not os.path.exists(gt_path):
            continue

        pred = torch.from_numpy(np.array(Image.open(os.path.join(render_dir, fname)).convert("L"))).float() > 0
        gt = torch.from_numpy(np.array(Image.open(gt_path).convert("L"))).float() > 0
        if gt.sum() == 0:
            continue

        intersection = torch.logical_and(pred, gt).sum().float()
        union = torch.logical_or(pred, gt).sum().float()
        iou = (intersection / union).item() * 100 if union > 0 else 0.0

        # 객체 이름 추출
        obj_name = fname.split(".png")[0]
        obj_name = ''.join([c for c in obj_name if not c.isdigit()]).strip()
        obj_type = OBJECT_TYPE.get(obj_name, "unknown")
        type_ious[obj_type].append(iou)

    return type_ious

# 전체 장면 합산
all_type_ious = {"transparent": [], "thin": [], "textured": [], "opaque": [], "unknown": []}

for scene in ALL_SCENES:
    scene_output = os.path.join(OUTPUT_ROOT, scene)
    r_dir = os.path.join(scene_output, "test_results", "renders")
    g_dir = os.path.join(scene_output, "test_results", "gt")
    if os.path.exists(r_dir) and os.path.exists(g_dir):
        result = analyze_object_type(r_dir, g_dir)
        for k, v in result.items():
            all_type_ious[k].extend(v)

# 시각화
types_with_data = {k: v for k, v in all_type_ious.items() if len(v) > 0}
type_names = list(types_with_data.keys())
type_means = [np.mean(types_with_data[t]) for t in type_names]
type_stds = [np.std(types_with_data[t]) for t in type_names]
type_counts = [len(types_with_data[t]) for t in type_names]

colors = {"transparent": "#3498db", "thin": "#e74c3c", "textured": "#f39c12", "opaque": "#2ecc71", "unknown": "#95a5a6"}
bar_colors = [colors.get(t, "#95a5a6") for t in type_names]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(type_names, type_means, yerr=type_stds, capsize=5, color=bar_colors, edgecolor='black', alpha=0.8)

# 막대 위에 개수 표시
for bar, count, mean in zip(bars, type_counts, type_means):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
            f'n={count}\n{mean:.1f}%', ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Mean IoU (%)', fontsize=12)
ax.set_title('Segmentation Performance by Object Type', fontsize=14)
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# 테이블 출력
print(f"\n{'유형':<15} {'개수':<8} {'평균 IoU':<12} {'표준편차':<12} {'최소':<8} {'최대':<8}")
print("-" * 63)
for t in type_names:
    v = types_with_data[t]
    print(f"{t:<15} {len(v):<8} {np.mean(v):<12.1f} {np.std(v):<12.1f} {np.min(v):<8.1f} {np.max(v):<8.1f}")

### 11-3. Referring Expression 유형별 성능 분석
JSON 어노테이션의 sentence를 분석하여, 공간 관계 표현(spatial)과 속성 표현(attribute)에 따른 성능 차이를 확인합니다.
- **Spatial**: "next to", "behind", "between", "left", "right", "near", "placed" 등
- **Attribute**: "red", "round", "small", "large", "colored" 등
- **Mixed**: 두 가지 모두 포함

In [ ]:
import os, json, re, torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from collections import defaultdict

SPATIAL_KEYWORDS = [
    "next to", "behind", "in front of", "between", "left", "right",
    "near", "close to", "placed", "above", "below", "on top of",
    "beside", "adjacent", "opposite", "facing", "under", "over",
    "resting on", "sitting on", "standing on"
]
ATTRIBUTE_KEYWORDS = [
    "red", "blue", "green", "yellow", "white", "black", "brown", "orange", "pink",
    "round", "square", "small", "large", "big", "tiny", "long", "short", "thin",
    "colored", "bright", "dark", "shiny", "surface", "flat", "wooden", "metal",
    "transparent", "glass"
]

def classify_sentence(sentence):
    """문장을 spatial / attribute / mixed / simple로 분류"""
    s = sentence.lower()
    has_spatial = any(kw in s for kw in SPATIAL_KEYWORDS)
    has_attribute = any(kw in s for kw in ATTRIBUTE_KEYWORDS)
    if has_spatial and has_attribute:
        return "mixed"
    elif has_spatial:
        return "spatial"
    elif has_attribute:
        return "attribute"
    else:
        return "simple"

def analyze_expression_type(data_root, output_root, scene_name):
    """Referring expression 유형별 IoU 분석"""
    scene_data = os.path.join(data_root, scene_name)
    scene_output = os.path.join(output_root, scene_name)
    r_dir = os.path.join(scene_output, "test_results", "renders")
    g_dir = os.path.join(scene_output, "test_results", "gt")

    if not os.path.exists(r_dir):
        print(f"  {scene_name}: 렌더링 결과 없음")
        return None

    # 테스트 JSON에서 sentence 로드
    json_dir = os.path.join(scene_data, "json", "test_json")
    if not os.path.isdir(json_dir):
        json_dir = os.path.join(scene_data, "test_json")
    if not os.path.isdir(json_dir):
        print(f"  {scene_name}: test_json 없음")
        return None

    # 카테고리 → (sentence, type) 매핑
    cat_sentences = defaultdict(list)
    for jf in sorted(os.listdir(json_dir)):
        if not jf.endswith(".json"):
            continue
        with open(os.path.join(json_dir, jf)) as f:
            data = json.load(f)
        for obj in data.get("object", []):
            cat = obj.get("category", "")
            for sent in obj.get("sentence", []):
                stype = classify_sentence(sent)
                cat_sentences[cat].append({"sentence": sent, "type": stype})

    # 렌더링 결과와 매칭
    type_ious = {"spatial": [], "attribute": [], "mixed": [], "simple": []}
    type_examples = {"spatial": [], "attribute": [], "mixed": [], "simple": []}

    for fname in sorted(os.listdir(r_dir)):
        if not fname.endswith(".png"):
            continue
        gt_path = os.path.join(g_dir, fname)
        if not os.path.exists(gt_path):
            continue

        pred = torch.from_numpy(np.array(Image.open(os.path.join(r_dir, fname)).convert("L"))).float() > 0
        gt = torch.from_numpy(np.array(Image.open(gt_path).convert("L"))).float() > 0
        if gt.sum() == 0:
            continue

        intersection = torch.logical_and(pred, gt).sum().float()
        union = torch.logical_or(pred, gt).sum().float()
        iou = (intersection / union).item() * 100 if union > 0 else 0.0

        # 카테고리 추출
        cat = fname.split(".png")[0]
        cat = ''.join([c for c in cat if not c.isdigit()]).strip()

        # 해당 카테고리의 sentence 유형으로 분류
        if cat in cat_sentences and cat_sentences[cat]:
            # 가장 많이 나오는 유형 사용
            types = [s["type"] for s in cat_sentences[cat]]
            dominant_type = max(set(types), key=types.count)
            type_ious[dominant_type].append(iou)
            if len(type_examples[dominant_type]) < 3:
                type_examples[dominant_type].append(cat_sentences[cat][0]["sentence"][:60])

    return type_ious, type_examples

# 전체 장면 분석
all_type_ious = {"spatial": [], "attribute": [], "mixed": [], "simple": []}
all_type_examples = {"spatial": [], "attribute": [], "mixed": [], "simple": []}

for scene in ALL_SCENES:
    print(f"\nAnalyzing {scene}...")
    result = analyze_expression_type(DATA_ROOT, OUTPUT_ROOT, scene)
    if result:
        type_ious, type_examples = result
        for k in all_type_ious:
            all_type_ious[k].extend(type_ious[k])
            all_type_examples[k].extend(type_examples[k])

# 시각화
types_with_data = {k: v for k, v in all_type_ious.items() if len(v) > 0}
if types_with_data:
    type_names = list(types_with_data.keys())
    type_means = [np.mean(types_with_data[t]) for t in type_names]
    type_stds = [np.std(types_with_data[t]) for t in type_names]
    type_counts = [len(types_with_data[t]) for t in type_names]

    colors = {"spatial": "#e74c3c", "attribute": "#3498db", "mixed": "#9b59b6", "simple": "#2ecc71"}
    bar_colors = [colors.get(t, "#95a5a6") for t in type_names]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(type_names, type_means, yerr=type_stds, capsize=5, color=bar_colors, edgecolor='black', alpha=0.8)
    for bar, count, mean in zip(bars, type_counts, type_means):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
                f'n={count}\n{mean:.1f}%', ha='center', va='bottom', fontsize=10)
    ax.set_ylabel('Mean IoU (%)', fontsize=12)
    ax.set_title('Performance by Referring Expression Type', fontsize=14)
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 테이블 + 예시
    print(f"\n{'유형':<12} {'개수':<8} {'평균 IoU':<12} {'예시 문장'}")
    print("-" * 80)
    for t in type_names:
        v = types_with_data[t]
        examples = all_type_examples.get(t, [])[:2]
        ex_str = " | ".join(examples) if examples else "-"
        print(f"{t:<12} {len(v):<8} {np.mean(v):<12.1f} {ex_str[:60]}")
else:
    print("분석할 데이터가 없습니다.")

### 11-4. 뷰 일관성 분석
동일 객체가 여러 테스트 뷰에서 얼마나 일관된 성능을 보이는지 분석합니다.
뷰에 따라 IoU 편차가 크다면, 3D scene knowledge 활용이 불충분하다는 증거입니다.

In [ ]:
import os, torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from collections import defaultdict

def analyze_view_consistency(render_dir, gt_dir, scene_name):
    """동일 객체의 뷰별 IoU 편차 분석"""
    obj_view_ious = defaultdict(list)  # {obj_name: [(view_id, iou), ...]}

    for fname in sorted(os.listdir(render_dir)):
        if not fname.endswith(".png"):
            continue
        gt_path = os.path.join(gt_dir, fname)
        if not os.path.exists(gt_path):
            continue

        pred = torch.from_numpy(np.array(Image.open(os.path.join(render_dir, fname)).convert("L"))).float() > 0
        gt = torch.from_numpy(np.array(Image.open(gt_path).convert("L"))).float() > 0
        if gt.sum() == 0:
            continue

        intersection = torch.logical_and(pred, gt).sum().float()
        union = torch.logical_or(pred, gt).sum().float()
        iou = (intersection / union).item() * 100 if union > 0 else 0.0

        # 뷰 ID와 객체 이름 추출 (00006bowl.png -> view=6, obj=bowl)
        base = fname.replace(".png", "")
        view_id = int(''.join([c for c in base if c.isdigit()]))
        obj_name = ''.join([c for c in base if not c.isdigit()]).strip()
        obj_view_ious[obj_name].append((view_id, iou))

    return obj_view_ious

# 전체 장면 분석
for scene in ALL_SCENES:
    scene_output = os.path.join(OUTPUT_ROOT, scene)
    r_dir = os.path.join(scene_output, "test_results", "renders")
    g_dir = os.path.join(scene_output, "test_results", "gt")
    if not os.path.exists(r_dir):
        continue

    print(f"\n{'='*60}")
    print(f"Scene: {scene} - View Consistency Analysis")
    print(f"{'='*60}")

    obj_view_ious = analyze_view_consistency(r_dir, g_dir, scene)

    # 2개 이상 뷰에 등장하는 객체만
    multi_view_objs = {k: v for k, v in obj_view_ious.items() if len(v) >= 2}

    if not multi_view_objs:
        print("  다중 뷰 객체 없음")
        continue

    # 객체별 IoU 통계
    obj_names = sorted(multi_view_objs.keys())
    obj_means = [np.mean([x[1] for x in multi_view_objs[o]]) for o in obj_names]
    obj_stds = [np.std([x[1] for x in multi_view_objs[o]]) for o in obj_names]
    obj_ranges = [np.max([x[1] for x in multi_view_objs[o]]) - np.min([x[1] for x in multi_view_objs[o]]) for o in obj_names]

    # 시각화: 각 객체의 뷰별 IoU 분포
    fig, ax = plt.subplots(figsize=(14, 6))
    positions = range(len(obj_names))
    for i, obj in enumerate(obj_names):
        ious = [x[1] for x in multi_view_objs[obj]]
        ax.scatter([i] * len(ious), ious, alpha=0.6, s=40, zorder=3)
        ax.plot([i, i], [min(ious), max(ious)], 'k-', alpha=0.3, linewidth=2)
        ax.plot(i, np.mean(ious), 'D', color='red', markersize=8, zorder=4)

    ax.set_xticks(list(positions))
    ax.set_xticklabels(obj_names, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('IoU (%)', fontsize=12)
    ax.set_title(f'{scene} - Per-Object IoU Across Views (red diamond = mean)', fontsize=13)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 테이블
    print(f"\n{'객체':<25} {'뷰 수':<8} {'평균 IoU':<10} {'표준편차':<10} {'범위':<10} {'안정성'}")
    print("-" * 73)
    for obj, mean, std, rng in zip(obj_names, obj_means, obj_stds, obj_ranges):
        stability = "안정" if std < 15 else ("불안정" if std < 30 else "매우 불안정")
        n_views = len(multi_view_objs[obj])
        print(f"{obj:<25} {n_views:<8} {mean:<10.1f} {std:<10.1f} {rng:<10.1f} {stability}")

    avg_std = np.mean(obj_stds)
    avg_range = np.mean(obj_ranges)
    print(f"\n평균 표준편차: {avg_std:.1f}%  |  평균 범위: {avg_range:.1f}%")

### 11-5. 실패 케이스 시각화
IoU가 가장 낮은 케이스와 가장 높은 케이스를 나란히 보여줍니다.
모델이 어떤 상황에서 실패하고 성공하는지 시각적으로 비교합니다.

In [ ]:
import os, torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image

def collect_all_results(output_root, scenes):
    """전체 결과를 (scene, filename, iou, pred_path, gt_path)로 수집"""
    all_results = []
    for scene in scenes:
        r_dir = os.path.join(output_root, scene, "test_results", "renders")
        g_dir = os.path.join(output_root, scene, "test_results", "gt")
        if not os.path.exists(r_dir):
            continue
        for fname in sorted(os.listdir(r_dir)):
            if not fname.endswith(".png"):
                continue
            gt_path = os.path.join(g_dir, fname)
            if not os.path.exists(gt_path):
                continue
            pred = torch.from_numpy(np.array(Image.open(os.path.join(r_dir, fname)).convert("L"))).float() > 0
            gt = torch.from_numpy(np.array(Image.open(gt_path).convert("L"))).float() > 0
            if gt.sum() == 0:
                continue
            intersection = torch.logical_and(pred, gt).sum().float()
            union = torch.logical_or(pred, gt).sum().float()
            iou = (intersection / union).item() * 100 if union > 0 else 0.0
            all_results.append({
                "scene": scene, "fname": fname, "iou": iou,
                "pred_path": os.path.join(r_dir, fname),
                "gt_path": gt_path
            })
    return sorted(all_results, key=lambda x: x["iou"])

results = collect_all_results(OUTPUT_ROOT, ALL_SCENES)
if not results:
    print("결과가 없습니다.")
else:
    n_show = 5

    # 최악 케이스
    worst = results[:n_show]
    # 최고 케이스
    best = results[-n_show:][::-1]

    fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))
    fig.suptitle('Worst Cases (left) vs Best Cases (right)', fontsize=16, y=1.01)

    for i in range(n_show):
        # Worst - Pred
        w_pred = np.array(Image.open(worst[i]["pred_path"]).convert("L"))
        w_gt = np.array(Image.open(worst[i]["gt_path"]).convert("L"))
        axes[i, 0].imshow(w_pred, cmap='gray')
        axes[i, 0].set_title(f"WORST Pred: {worst[i]['scene']}/{worst[i]['fname']}\nIoU: {worst[i]['iou']:.1f}%", fontsize=8)
        axes[i, 0].axis('off')
        axes[i, 1].imshow(w_gt, cmap='gray')
        axes[i, 1].set_title(f"GT", fontsize=8)
        axes[i, 1].axis('off')

        # Best - Pred
        b_pred = np.array(Image.open(best[i]["pred_path"]).convert("L"))
        b_gt = np.array(Image.open(best[i]["gt_path"]).convert("L"))
        axes[i, 2].imshow(b_pred, cmap='gray')
        axes[i, 2].set_title(f"BEST Pred: {best[i]['scene']}/{best[i]['fname']}\nIoU: {best[i]['iou']:.1f}%", fontsize=8)
        axes[i, 2].axis('off')
        axes[i, 3].imshow(b_gt, cmap='gray')
        axes[i, 3].set_title(f"GT", fontsize=8)
        axes[i, 3].axis('off')

    plt.tight_layout()
    plt.show()

    # IoU 분포 히스토그램
    all_ious = [r["iou"] for r in results]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(all_ious, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(all_ious), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(all_ious):.1f}%')
    ax.axvline(np.median(all_ious), color='orange', linestyle='--', linewidth=2, label=f'Median: {np.median(all_ious):.1f}%')
    ax.set_xlabel('IoU (%)', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('IoU Distribution Across All Scenes', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 통계 요약
    print(f"\n전체 쿼리 수: {len(all_ious)}")
    print(f"Mean IoU: {np.mean(all_ious):.1f}%")
    print(f"Median IoU: {np.median(all_ious):.1f}%")
    print(f"IoU = 0% 비율: {sum(1 for x in all_ious if x < 1) / len(all_ious) * 100:.1f}%")
    print(f"IoU > 50% 비율: {sum(1 for x in all_ious if x > 50) / len(all_ious) * 100:.1f}%")

### 11-6. 종합 분석 요약
위 실험들의 결과를 하나의 테이블로 정리합니다.

In [ ]:
print("=" * 70)
print("ReferSplat 문제점 분석 종합 요약")
print("=" * 70)

print("""
[문제 1] 객체 크기 의존성
  - 큰 객체(>15% 면적): 높은 IoU
  - 작은 객체(<1% 면적): 거의 0% IoU
  → 16차원 referring feature가 작은 객체의 세밀한 특징을 표현하기 어려움
  → Gaussian 포인트 밀도가 작은 객체에서 부족

[문제 2] 객체 유형별 한계
  - 투명/반사 객체: 3DGS 자체가 복원 못하므로 세그멘테이션 불가
  - 가늘고 긴 객체: Gaussian 표현의 근본적 한계
  → 3DGS 기반 방법의 구조적 한계

[문제 3] 공간 관계 이해 부족
  - Spatial 표현 포함 문장의 성능이 낮을 경우:
    Position-aware Cross-Modal Interaction이 실제 공간 관계를
    충분히 모델링하지 못하고 있음을 시사
  → 논문의 핵심 기여에 대한 의문 제기 가능

[문제 4] 뷰 일관성 부족
  - 동일 객체의 뷰별 IoU 편차가 큰 경우:
    3D scene knowledge 활용이 불충분
  - 특정 뷰에서만 잘 되고 다른 뷰에서 실패
  → multi-view consistency 강화 필요

[문제 5] 성능 양극화
  - IoU 분포가 bimodal (0% 근처 + 80% 이상)
  - "되는 건 잘 되고, 안 되는 건 아예 안 됨"
  → pseudo mask 품질이 성능 상한선을 결정하는 병목
""")

print("=" * 70)